In [5]:
!pip install torch transformers rdflib nltk scikit-learn tqdm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import json
import re
import numpy as np
from rdflib import Graph, Namespace, RDF, Literal, URIRef, OWL
from sklearn.metrics.pairwise import cosine_similarity
import logging
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# NLTK downloads
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Preprocessing utilities
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Load DeBERTa
MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ssanj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ssanj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ssanj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


DebertaV2Model(
  (embeddings): DebertaV2Embeddings(
    (word_embeddings): Embedding(128100, 768, padding_idx=0)
    (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): DebertaV2Encoder(
    (layer): ModuleList(
      (0-5): 6 x DebertaV2Layer(
        (attention): DebertaV2Attention(
          (self): DisentangledSelfAttention(
            (query_proj): Linear(in_features=768, out_features=768, bias=True)
            (key_proj): Linear(in_features=768, out_features=768, bias=True)
            (value_proj): Linear(in_features=768, out_features=768, bias=True)
            (pos_dropout): Dropout(p=0.1, inplace=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): DebertaV2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
            (dropout): Dropout(p=0.1, in

In [7]:
def preprocess(text):
    text = re.sub(r'[_\-\.]', ' ', text.lower())
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

def get_embedding(text):
    if not text.strip():
        return np.zeros(model.config.hidden_size)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()[0]
    return embedding


In [8]:
def parse_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    fields = set()
    def extract(obj, path=""):
        if isinstance(obj, dict):
            for k, v in obj.items():
                fields.add(k)
                if isinstance(v, (dict, list)):
                    extract(v, path + f".{k}" if path else k)
        elif isinstance(obj, list):
            for item in obj:
                extract(item, path)
    extract(data)
    return list(fields)

def parse_owl(owl_file):
    g = Graph()
    try:
        g.parse(owl_file, format="xml")
    except Exception:
        try:
            g.parse(owl_file, format="turtle")
        except Exception:
            g.parse(owl_file, format="json-ld")

    terms = {}
    for s, p, o in g.triples((None, RDF.type, OWL.Class)):
        terms[s.split("#")[-1]] = str(s)
    for s, p, o in g.triples((None, RDF.type, OWL.ObjectProperty)):
        terms[s.split("#")[-1]] = str(s)
    for s, p, o in g.triples((None, RDF.type, OWL.DatatypeProperty)):
        terms[s.split("#")[-1]] = str(s)
    return terms


In [9]:
def map_fields(json_fields, ontology_terms_dict, threshold=0.7):
    results = []
    ontology_embeddings = {}

    print("Embedding ontology terms...")
    for term_name, uri in tqdm(ontology_terms_dict.items()):
        embedding = get_embedding(preprocess(term_name))
        ontology_embeddings[term_name] = (embedding, uri)

    print("Mapping JSON fields...")
    for field in tqdm(json_fields):
        field_vector = get_embedding(preprocess(field))
        best_match, best_score, best_uri = None, 0.0, None

        for term_name, (emb, uri) in ontology_embeddings.items():
            if np.any(field_vector) and np.any(emb):
                score = cosine_similarity(field_vector.reshape(1, -1), emb.reshape(1, -1))[0][0]
            else:
                score = 0.0
            if score > best_score:
                best_score, best_match, best_uri = score, term_name, uri

        results.append({
            "field": field,
            "match": best_match,
            "match_uri": best_uri,
            "score": best_score,
            "mapped": best_score >= threshold
        })
    return results


In [10]:
def generate_owl(mapped_results, output_file):
    g = Graph()
    NS = Namespace("https://orbis-security.com/pe-malware-ontology#")
    g.bind("mapped", NS)
    g.bind("pe", NS)

    for idx, res in enumerate(mapped_results):
        ind = NS[f"Sample{idx+1}"]
        g.add((ind, RDF.type, NS.MappedEntity))
        g.add((ind, NS.originalField, Literal(res["field"])))
        g.add((ind, NS.mappedTo, URIRef(res["match_uri"]) if res["match_uri"] else Literal(res["match"] or "None")))
        g.add((ind, NS.similarityScore, Literal(res["score"])))

    g.serialize(destination=output_file, format="xml")
    print(f"OWL file written to {output_file}")


In [11]:
json_file = "D:/study/nlp_mapping/00a35f1e23cef590bdfd8d2d30ecd7024b0028e34433384a840bf37638647af1.json"
owl_file = "D:/study/nlp_mapping/pe_malware_ontology.owl"
output_file = "D:/study/nlp_mapping/mapped_output.owl"

json_fields = parse_json(json_file)
ontology_terms_dict = parse_owl(owl_file)
mapped_results = map_fields(json_fields, ontology_terms_dict, threshold=0.7)

# show results live in notebook
for res in mapped_results:
    status = "✔" if res["mapped"] else "✘"
    print(f"{res['field']} → {res['match']} ({res['score']:.2f}) {status}")

generate_owl(mapped_results, output_file)


Embedding ontology terms...


  0%|          | 0/209 [00:00<?, ?it/s]

Mapping JSON fields...


  0%|          | 0/283 [00:00<?, ?it/s]

sublanguage → KillThread (0.97) ✔
T1129 → CLR (0.96) ✔
duration → Executable (0.99) ✔
iplookups → EnumerateThreads (0.98) ✔
aux_sha1 → url_strings_count (0.95) ✔
aux_valid → has_section_feature (0.97) ✔
calls → Action (0.98) ✔
threads → Symbols (0.99) ✔
started_services → has_file_feature (0.98) ✔
pathtofile → WriteToFile (0.99) ✔
ra → None (0.00) ✘
parent_sample → has_section_flag (0.99) ✔
eve_log_full_path → url_strings_count (0.96) ✔
new_data → has_file_feature (0.98) ✔
copy → Signature (0.99) ✔
rrname → PEFile (0.99) ✔
tx_id → None (0.00) ✘
command → Action (0.99) ✔
license → Executable (0.99) ✔
statistics → Symbols (0.99) ✔
api → Debug (0.99) ✔
size → Executable (0.99) ✔
Hit → Action (0.99) ✔
entrypoint → CodeSection (0.98) ✔
files → Resources (0.99) ✔
size_of_data → has_file_feature (0.98) ✔
configs → section_name (0.98) ✔
payloads → Action (0.99) ✔
imphash → PEFile (0.99) ✔
tls → TLS (1.00) ✔
from → None (0.00) ✘
write_files → section_name (0.98) ✔
name → Symbols (0.99) ✔
clamav